# Loan default: baseline models
**Goal:** set a baseline score that later models must beat.
**Data:** 255,347 loans loaded from Postgres, 16 features after dropping `LoanID` and the target `Default`.
**Approach:** stratified 80/20 train/test split, numeric features only (12 columns), three models: a dummy, logistic regression, and logistic regression with balanced class weights.

## 1. Imports and data
Load the `loans` table from Postgres. The three Yes/No columns are already stored as 1/0.

In [12]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
from loan_data import read_from_db
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, precision_score

df = read_from_db()
print(df.shape)
df.dtypes

(255347, 18)


LoanID                str
Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education             str
EmploymentType        str
MaritalStatus         str
HasMortgage         int64
HasDependents       int64
LoanPurpose           str
HasCoSigner         int64
Default             int64
dtype: object

## 2. Features, target and stratified split
`X` holds the features and `y` the target `Default`. `LoanID` is dropped because it only identifies a row. The split is stratified, so both parts keep the same default rate, and `random_state=42` makes it repeatable. The test set stays untouched until evaluation.

In [6]:
X = df.drop(columns=["LoanID", "Default"])
y = df["Default"]

X_train, X_test, y_train, y_test=train_test_split(
    X, y, test_size = 0.2, stratify=y, random_state=42
)
print(X_train.shape,X_test.shape)
print("Default rate: ", y_train.mean())
print("test: ", y_test.mean())

(204277, 16) (51070, 16)
Default rate:  0.11612663197521013
test:  0.11613471705502251


**Check:** 204,277 training rows and 51,070 test rows. Default rate is 11.61% in both, so the stratified split worked.

## 3. Dummy baseline
A model that always predicts "no default". It shows the accuracy that needs no skill at all.

In [7]:
num_cols = X.select_dtypes("number").columns.to_list()
print(len(num_cols), num_cols)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train[num_cols], y_train)
dummy_pred = dummy.predict(X_test[num_cols])
print("Dummy accuracy: ", accuracy_score(y_test, dummy_pred))
print("Dummy recall: ", recall_score(y_test, dummy_pred))

12 ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'HasMortgage', 'HasDependents', 'HasCoSigner']
Dummy accuracy:  0.8838652829449775
Dummy recall:  0.0


**Finding:** the dummy scores 88.4% accuracy with 0% recall. High accuracy here means nothing, because it simply equals the share of repaid loans. This is why I judge models on recall, precision, F1 and ROC-AUC.

## 4. Logistic regression (numeric features only)
Scaling and the model sit in one pipeline, so the scaler learns only from the training data. Text columns (education, employment type, marital status, loan purpose) are left out for now and will be encoded in a pipeline on Day 10.

In [8]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train[num_cols], y_train)

train_pred = model.predict(X_train[num_cols])
test_pred = model.predict(X_test[num_cols])
print("train accuracy: ",accuracy_score(y_train, train_pred))
print("test accuracy: ",accuracy_score(y_test, test_pred))
print("test recall: ",recall_score(y_test, test_pred))
print(confusion_matrix(y_test, test_pred))

train accuracy:  0.8849062792189037
test accuracy:  0.8845310358331702
test recall:  0.023604788399932557
[[45033   106]
 [ 5791   140]]


**Finding:** test accuracy is 88.5% against 88.4% for the dummy, and recall is only 2.4% (140 of 5,931 defaults caught). Train accuracy (88.5%) is almost the same as test accuracy, so the model is not overfitting; it is just predicting "no default" almost every time.

| | Predicted repaid | Predicted default |
|---|---|---|
| **Actually repaid** | 45,033 | 106 |
| **Actually defaulted** | 5,791 | 140 |

## 5. Logistic regression with balanced class weights
`class_weight="balanced"` makes mistakes on the rare default class cost more, so the model tries harder to find defaults.

In [5]:
model_bal = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight = "balanced"))
model_bal.fit(X_train[num_cols],y_train)

bal_pred = model_bal.predict(X_test[num_cols])
print("Accuracy: ", accuracy_score(y_test,bal_pred))
print("recall: ", recall_score(y_test,bal_pred))
print(confusion_matrix(y_test,bal_pred))

Accuracy:  0.6718817309575092
recall:  0.6990389479008599
[[30167 14972]
 [ 1785  4146]]


**Finding:** recall jumps from 2.4% to 69.9% (4,146 of 5,931 defaults caught), but accuracy drops to 67.2%. The price is 14,972 repaid loans flagged as risky, so only 21.7% of flagged loans really default. This trade-off between catching defaults and false alarms is the main decision for the next stage.

In [16]:
preds = {
    "Dummy (always no default)":dummy_pred,
    "Logistic Regression":test_pred,
    "Logistic, Balanced weights":bal_pred
}
results = pd.DataFrame({
    name: {
        "Accuracy": accuracy_score(y_test, p),
        "Recall": recall_score(y_test, p),
        "Precision": precision_score(y_test, p, zero_division=0),
    } 
    for name, p in preds.items()
}).T.round(3)
results

,Accuracy,Recall,Precision
Dummy (always no default),0.884,0.000,0.000
Logistic Regression,0.885,0.024,0.569
"Logistic, Balanced weights",0.672,0.699,0.217


## Baseline results (numeric features only)
| Model | Accuracy | Recall | Precision |
|---|---|---|---|
| Dummy (always "no default") | 0.884 | 0.000 | 0.000 |
| Logistic regression | 0.885 | 0.024 | 0.569 |
| Logistic regression, balanced weights | 0.672 | 0.699 | 0.217 |

**Takeaways**
- Accuracy alone is misleading on this imbalanced data.
- Plain logistic regression barely beats the dummy.
- Class weights raise recall sharply but cost precision and accuracy.
- Next: add precision, F1 and ROC-AUC, encode the text columns, and try tree-based models.